>  이 파일은 해설 버전입니다.

# 2. OpenSearch 실습 - 환경 설정 및 인덱싱.....!!!

## 학습 목표
- OpenSearch의 인덱스 개념을 이해한다
- 텍스트 데이터를 벡터화하여 저장하는 과정을 실습한다
- 문서 전처리 및 청킹(Chunking) 기법을 학습한다

## 2.1 OpenSearch 환경 설정
### OpenSearch 실습 환경 접속 가이드

1. 필요 패키지 설치

```bash
pip install opensearch-py openai
```

2. OpenSearch 서버 접속 설정

```python
from opensearchpy import OpenSearch

# ★ 본인 번호로 변경 (0~30) ★
STUDENT_NUMBER = 0

# 서버 접속 정보
OPENSEARCH_HOST = "sdsos.baeum.ai.kr"
OPENSEARCH_PORT = 443
OPENSEARCH_USER = "sdsrag"
OPENSEARCH_PASS = "Student.Rag1!"

# 클라이언트 생성
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=True,
    verify_certs=True,
    ssl_show_warn=False,
)

# 접속 확인
print(client.info()["version"]["number"])
```

3. 인덱스 이름 규칙

```python
# 본인 번호에 맞는 인덱스 사용 (다른 번호 인덱스 사용 금지)
INDEX_NAME = f"student_{STUDENT_NUMBER:02d}_company_docs"
# 예: student_01_company_docs, student_02_company_docs, ...
```

4. 접속 확인 테스트

```python
# 클러스터 상태 확인
print(client.cluster.health()["status"])

# 내 인덱스 목록 확인
print(client.cat.indices(index="student_*", format="json"))
```

5. OpenSearch Dashboards (웹 UI)

- 주소: https://sdsos.baeum.ai.kr/dashboard
- 계정: sdsrag / Student.Rag1!
- 로그인 후 좌측 메뉴 → Discover → student_* 인덱스 패턴 선택
- 노트북에서 인덱싱한 문서를 검색/시각화할 수 있음

6. 주의사항

- STUDENT_NUMBER를 반드시 본인 번호로 변경할 것
- 다른 학생의 인덱스(student_XX_*)에는 접근할 수 없음
- OpenAI API 키는 각자 본인 키를 사용할 것

In [ ]:
# [02-01] 필요 패키지 설치......
# [목적] OpenSearch 연결과 OpenAI API 호출에 필요한 라이브러리를 설치합니다
# [주의] 처음 한 번만 실행합니다. 오류 시 [런타임 → 다시 시작] 후 재실행하세요
# 필요 패키지 설치
!pip install -q opensearch-py openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 4.5 MB/s eta 0:00:00


In [ ]:
# [02-02] 라이브러리 임포트
# [목적] OpenSearch 서버와 통신하기 위한 클라이언트 모듈을 불러옵니다
# [개념] opensearchpy = Python에서 OpenSearch 검색엔진과 통신하는 공식 라이브러리
from opensearchpy import OpenSearch

In [ ]:
# [02-03] OpenSearch 접속 정보 설정
# [목적] 실습용 OpenSearch 서버의 주소, 계정, 포트를 변수로 설정합니다
# [주의] STUDENT_NUMBER를 반드시 본인 번호(0~30)로 변경하세요! 다른 번호는 접근 불가
# ★ 본인 번호로 변경 (0~30) ★
STUDENT_NUMBER = 0

# 서버 접속 정보
OPENSEARCH_HOST = "sdsos.baeum.ai.kr"  # OpenSearch 서버 도메인
OPENSEARCH_PORT = 443                   # HTTPS 기본 포트
OPENSEARCH_USER = "sdsrag"              # 실습 공용 계정
OPENSEARCH_PASS = "Student.Rag1!"       # 실습 공용 비밀번호

In [ ]:
# [02-04] OpenSearch 연결 확인
# [목적] 설정한 정보로 OpenSearch 서버에 실제 연결하고 버전을 확인합니다
# [주의] 여기서 오류 나면 이후 모든 셀이 실패합니다. 호스트/계정/포트를 재확인하세요
# 클라이언트 생성
opensearch_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],  # 접속 대상 호스트:포트
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),  # (아이디, 비밀번호) 튜플
    use_ssl=True,        # HTTPS 사용
    verify_certs=True,   # SSL 인증서 검증
    ssl_show_warn=False,  # SSL 경고 메시지 숨김
)

# 접속 확인: 서버 버전 출력으로 연결 성공 여부 판별
print(opensearch_client.info()["version"]["number"])

2.19.4


In [ ]:
# [02-05] OpenAI 클라이언트 생성
# [목적] 임베딩 생성에 사용할 OpenAI API 클라이언트를 초기화합니다
# [주의] 환경변수에 OPENAI_API_KEY가 설정되어 있어야 합니다
from openai import OpenAI

# OpenAI API 키는 각자 본인 키를 사용
# 예:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
openai_client = OpenAI()  # 환경변수 OPENAI_API_KEY를 자동으로 읽어 인증

In [ ]:
# [02-06] 인덱스 이름 및 임베딩 설정
# [목적] 본인 전용 인덱스 이름과 임베딩 모델/차원 수를 설정합니다
# [개념] 인덱스 = 문서들의 저장소 (DB의 테이블과 유사). 학생마다 고유 인덱스를 사용합니다
# 본인 번호에 맞는 인덱스 사용 (다른 번호 인덱스 사용 금지)
INDEX_NAME = f"student_{STUDENT_NUMBER:02d}_company_docs"  # :02d → 두 자리 zero-padding
# 예: student_01_company_docs, student_02_company_docs, ...

EMBEDDING_MODEL = "text-embedding-3-small"  # OpenAI 경량 임베딩 모델
EMBEDDING_DIM = 1536                        # 해당 모델의 벡터 차원 수

In [ ]:
# [02-07] 클러스터 상태 확인
# [목적] OpenSearch 클러스터 상태와 기존 인덱스 목록을 확인합니다
# [개념] health가 green/yellow이면 정상, red이면 문제가 있는 상태입니다
# 클러스터 상태 확인 (green/yellow=정상, red=장애)
print(opensearch_client.cluster.health()["status"])

# student_로 시작하는 모든 인덱스 목록 확인
print(opensearch_client.cat.indices(index="student_*", format="json"))

print(f"사용 인덱스: {INDEX_NAME}")

yellow
[{'health': 'green', 'status': 'open', 'index': 'student_00_company_docs', 'uuid': 'JjMuaq8xQrqcBd7bp5vY7g', 'pri': '1', 'rep': '0', 'docs.count': '11', 'docs.deleted': '0', 'store.size': '426.2kb', 'pri.store.size': '426.2kb'}]
사용 인덱스: student_00_company_docs


## 2.2 인덱스(Index) 이해하기

### 인덱스란?
- 문서(Document)들의 논리적 집합
- RDBMS의 테이블과 유사한 개념
- 각 인덱스는 고유한 매핑(스키마)을 가짐

### 인덱스 구성 요소

### 필드 타입

| 타입 | 용도 | 예시 |
|------|------|------|
| text | 전문 검색 (토큰화됨) | 문서 내용, 제목 |
| keyword | 정확한 매칭, 집계 | 카테고리, 태그, ID |
| integer/float | 숫자 | 가격, 조회수 |
| date | 날짜/시간 | 생성일, 수정일 |
| boolean | 참/거짓 | 공개 여부 |
| knn_vector | 벡터 | 임베딩 |

## 2.3 인덱스 생성

In [ ]:
# [02-08] 인덱스 이름 확인
# [목적] 현재 설정된 인덱스 이름을 출력하여 올바른지 확인합니다
# 설정
print(f"사용 인덱스: {INDEX_NAME}")

사용 인덱스: student_00_company_docs


In [ ]:
# [02-09] 인덱스 스키마 정의
# [목적] 문서를 저장할 인덱스의 구조(필드 타입, 벡터 설정)를 JSON으로 정의합니다
# [개념] 매핑(mapping) = DB 스키마. text(전문검색용), keyword(필터용), knn_vector(벡터검색용) 타입 지정
# 인덱스 스키마 정의 (settings: 인덱스 설정, mappings: 필드 타입 정의)
index_body = {
    "settings": {
        "index": {
            "number_of_shards": 1,    # 데이터 분할 수 (실습: 1개, 운영: 규모에 따라 조정)
            "number_of_replicas": 0,  # 복제본 수 (실습: 0, 운영: 1 이상 권장)
            "knn": True,              # k-NN 벡터 검색 플러그인 활성화
        }
    },
    "mappings": {
        "properties": {
            # 텍스트 필드: 토큰화되어 전문 검색(Full-Text Search) 가능
            "title": {
                "type": "text",        # 전문 검색용 타입
                "analyzer": "standard",  # 표준 토크나이저 (공백+소문자 변환)
            },
            "content": {
                "type": "text",        # 본문 전문 검색용
                "analyzer": "standard",
            },
            # 메타데이터 필드: keyword 타입은 토큰화 없이 정확한 값으로 필터링/집계
            "category": {"type": "keyword"},    # 카테고리 필터용
            "source": {"type": "keyword"},      # 출처 필터용
            "created_at": {"type": "date"},     # 날짜 범위 검색용
            # 벡터 필드: 임베딩 벡터를 저장하고 k-NN 검색에 사용
            "embedding": {
                "type": "knn_vector",            # 벡터 검색 전용 타입
                "dimension": EMBEDDING_DIM,      # 벡터 차원 (1536)
                "method": {
                    "name": "hnsw",              # 근사 최근접 이웃 탐색 알고리즘
                    "space_type": "l2",          # 유클리드 거리 (운영에서는 cosinesimil도 사용)
                    "engine": "lucene",          # 벡터 인덱싱 엔진 (lucene 또는 nmslib)
                },
            },
        }
    },
}

In [ ]:
# [02-10] 인덱스 생성 함수
# [목적] 정의한 스키마로 OpenSearch에 인덱스를 실제로 생성하는 함수입니다
# [개념] recreate=True → 기존 인덱스 삭제 후 재생성. 실습 중 초기화할 때 유용합니다
# 인덱스 생성 함수
def create_index(index_name: str, body: dict, recreate: bool = False):
    """인덱스 생성"""
    if opensearch_client.indices.exists(index=index_name):  # 인덱스 존재 여부 확인
        if recreate:
            opensearch_client.indices.delete(index=index_name)  # 기존 인덱스 삭제
            print(f"기존 인덱스 '{index_name}' 삭제")
        else:
            print(f"인덱스 '{index_name}' 이미 존재")
            return  # 재생성하지 않으면 조기 종료

    opensearch_client.indices.create(index=index_name, body=body)  # 스키마 적용하여 인덱스 생성
    print(f"인덱스 '{index_name}' 생성 완료")

# 인덱스 생성 (recreate=True: 기존 인덱스 삭제 후 재생성)
create_index(INDEX_NAME, index_body, recreate=True)

기존 인덱스 'student_00_company_docs' 삭제
인덱스 'student_00_company_docs' 생성 완료


In [ ]:
# [02-11] 인덱스 매핑 확인
# [목적] 생성된 인덱스의 필드 타입이 의도대로 설정되었는지 확인합니다
# [개념] title(text), category(keyword), embedding(knn_vector) 등이 보이면 성공
# 인덱스 매핑 확인: 각 필드의 타입이 의도대로 생성되었는지 검증
mapping = opensearch_client.indices.get_mapping(index=INDEX_NAME)  # 매핑 정보 조회
print("인덱스 매핑:")
for field, config in mapping[INDEX_NAME]["mappings"]["properties"].items():  # 필드별 순회
    field_type = config.get("type", "object")  # 타입 정보 추출 (없으면 object)
    print(f"  {field}: {field_type}")

인덱스 매핑:
  category: keyword
  content: text
  created_at: date
  embedding: knn_vector
  source: keyword
  title: text


## 2.4 임베딩 생성 함수

In [ ]:
# [02-12] get_embedding 함수 정의
# [목적] 텍스트 하나를 OpenAI API로 임베딩 벡터로 변환하는 함수입니다
# [개념] 반환되는 1536차원 벡터가 OpenSearch의 knn_vector 필드에 저장됩니다
def get_embedding(text: str) -> list[float]:
    """OpenAI API를 사용한 텍스트 임베딩 생성"""
    response = openai_client.embeddings.create(
        input=text,                  # 임베딩할 텍스트
        model=EMBEDDING_MODEL,       # 사용할 모델 (text-embedding-3-small)
        dimensions=EMBEDDING_DIM,    # 출력 벡터 차원 (1536)
    )
    return response.data[0].embedding  # 응답에서 첫 번째 임베딩 벡터 추출

# 테스트
test_emb = get_embedding("테스트 문장입니다")
print(f"임베딩 차원: {len(test_emb)}")

임베딩 차원: 1536


In [ ]:
# [02-13] get_embeddings_batch 함수 정의
# [목적] 여러 텍스트를 한 번의 API 호출로 일괄 임베딩하는 배치 함수입니다
# [개념] 배치 처리 = 건별 호출보다 빠르고 효율적 (API 호출 횟수 감소)
def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    """배치 임베딩 생성 (효율적)"""
    response = openai_client.embeddings.create(
        input=texts,                 # 여러 텍스트를 리스트로 전달 (한 번에 처리)
        model=EMBEDDING_MODEL,       # 사용할 모델
        dimensions=EMBEDDING_DIM,    # 출력 벡터 차원
    )
    return [item.embedding for item in response.data]  # 각 텍스트의 임베딩을 리스트로 반환

# 테스트: 3개 문장을 한 번에 임베딩
test_texts = ["첫 번째 문장", "두 번째 문장", "세 번째 문장"]
test_embs = get_embeddings_batch(test_texts)
print(f"배치 임베딩 수: {len(test_embs)}")

배치 임베딩 수: 3


## 2.5 문서 전처리 및 청킹(Chunking)
### 청킹이란?
긴 문서를 작은 조각(chunk)으로 나누는 과정

### 청킹이 필요한 이유
1. 임베딩 모델의 토큰 제한 (보통 8K 토큰)
2. 검색 정확도 향상 (관련 부분만 검색)
3. LLM 컨텍스트 윈도우 효율적 사용

### 청킹 전략

| 전략 | 설명 | 장점 | 단점 |
|------|------|------|------|
| 고정 크기 | N자 또는 N토큰 단위 | 간단 | 문맥 단절 |
| 문단 기반 | 문단 단위로 분할 | 문맥 유지 | 크기 불균일 |
| 문장 기반 | 문장 단위로 분할 | 의미 단위 | 너무 작을 수 있음 |
| 오버랩 | 청크 간 중복 포함 | 문맥 연결 | 저장 공간 증가 |

In [ ]:
# [02-14] chunk_text 함수 정의
# [목적] 긴 텍스트를 고정 크기 조각(chunk)으로 나누는 함수입니다
# [개념] 청킹 = 긴 문서를 검색에 적합한 크기로 분할. overlap으로 문맥 끊김을 방지합니다
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """텍스트를 고정 크기로 청킹 (오버랩 포함)"""
    chunks = []  # 생성된 청크를 담을 리스트
    start = 0    # 현재 읽기 시작 위치

    while start < len(text):  # 텍스트 끝까지 반복
        end = start + chunk_size  # 청크 끝 위치 계산
        chunk = text[start:end]   # 텍스트 슬라이싱으로 청크 추출

        # 단어 중간에서 잘리지 않도록 조정
        if end < len(text):  # 마지막 청크가 아닌 경우에만
            # 마지막 공백 위치 찾기
            last_space = chunk.rfind(' ')  # 뒤에서부터 공백 검색
            if last_space > chunk_size * 0.8:  # 청크의 80% 이후에 공백이 있으면
                chunk = chunk[:last_space]     # 공백 직전까지만 사용
                end = start + last_space       # 끝 위치 재조정

        chunks.append(chunk.strip())  # 양쪽 공백 제거 후 저장
        start = end - overlap  # 다음 시작점 = 현재 끝 - 오버랩 (문맥 연결)

    return chunks

In [ ]:
# [02-15] 청킹 테스트
# [목적] chunk_text 함수가 올바르게 동작하는지 샘플 텍스트로 확인합니다
# [개념] chunk_size=150, overlap=30일 때 각 청크의 크기와 겹치는 부분을 확인합니다
# 청킹 테스트: 샘플 텍스트를 150자 단위로 분할, 30자 오버랩
sample_text = """인공지능(AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현한 컴퓨터 시스템입니다.
머신러닝은 AI의 한 분야로, 데이터를 통해 패턴을 학습하고 예측을 수행합니다.
딥러닝은 머신러닝의 한 종류로, 인공신경망을 사용하여 복잡한 패턴을 학습합니다.
최근에는 대규모 언어 모델(LLM)이 주목받고 있으며, GPT, Claude, Gemini 등이 대표적입니다.
이러한 모델들은 자연어 처리, 코드 생성, 질의응답 등 다양한 태스크를 수행할 수 있습니다."""

chunks = chunk_text(sample_text, chunk_size=150, overlap=30)  # 150자 청크, 30자 중복
print(f"총 {len(chunks)}개 청크 생성:\n")
for i, chunk in enumerate(chunks):
    print(f"[청크 {i+1}] ({len(chunk)}자)")
    print(f"{chunk}\n")

총 3개 청크 생성:

[청크 1] (138자)
인공지능(AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현한 컴퓨터 시스템입니다. 
머신러닝은 AI의 한 분야로, 데이터를 통해 패턴을 학습하고 예측을 수행합니다. 
딥러닝은 머신러닝의 한 종류로, 인공신경망을 사용하여 복잡한 패턴을

[청크 2] (145자)
러닝의 한 종류로, 인공신경망을 사용하여 복잡한 패턴을 학습합니다.
최근에는 대규모 언어 모델(LLM)이 주목받고 있으며, GPT, Claude, Gemini 등이 대표적입니다.
이러한 모델들은 자연어 처리, 코드 생성, 질의응답 등 다양한 태스크를 수행할 수

[청크 3] (36자)
, 코드 생성, 질의응답 등 다양한 태스크를 수행할 수 있습니다.



## 2.6 문서 색인(Indexing)

### 색인 과정

### sample_data 사전 점검

Colab/로컬에서 `sample_data.py` 파일이 없으면 이후 인덱싱 단계가 실패합니다.
아래 셀로 파일 존재 여부와 데이터 구조를 먼저 확인하세요.

In [ ]:
# [02-16] sample_data 파일 사전 점검
# [목적] 인덱싱에 사용할 sample_data.py 파일이 존재하고 올바른 형식인지 확인합니다
# [주의] 파일이 없으면 이후 인덱싱이 실패합니다. Colab에서는 파일 업로드가 필요합니다
from pathlib import Path
import importlib.util

sample_path = Path('sample_data.py')  # 확인할 파일 경로
print('sample_data.py 존재:', sample_path.exists())

if sample_path.exists():
    # importlib로 동적 모듈 로드 (import 없이 파일에서 직접 불러오기)
    spec = importlib.util.spec_from_file_location('sample_data', sample_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)  # 모듈 실행하여 변수/함수 로드
    docs = getattr(module, 'SAMPLE_DOCS', None)  # SAMPLE_DOCS 변수 가져오기
    if isinstance(docs, list) and docs:
        print('문서 개수:', len(docs))
        print('첫 문서 키:', list(docs[0].keys()))  # 문서 구조 확인
    else:
        print('SAMPLE_DOCS 형식을 확인하세요.')
else:
    print('현재 작업 디렉터리에 sample_data.py를 업로드/복사하세요.')

sample_data.py 존재: True
SAMPLE_DOCS 형식을 확인하세요.


In [ ]:
# [02-17] 샘플 데이터 로드
# [목적] sample_data.py에서 샘플 문서 리스트를 불러와 카테고리별 문서 수를 확인합니다
# sample_data.py에서 샘플 데이터 가져오기
from sample_data import SAMPLE_DOCUMENTS

print(f"총 {len(SAMPLE_DOCUMENTS)}개 문서 로드")
# 카테고리별 문서 수를 집계하여 출력
print("\n카테고리별 문서 수:")
categories = {}  # 카테고리: 개수 딕셔너리
for doc in SAMPLE_DOCUMENTS:
    cat = doc["category"]
    categories[cat] = categories.get(cat, 0) + 1  # 카운트 증가 (없으면 0부터)
for cat, count in sorted(categories.items()):  # 알파벳순 정렬 출력
    print(f"  - {cat}: {count}개")

총 20개 문서 로드

카테고리별 문서 수:
  - AI/ML: 2개
  - IT: 3개
  - 교육: 2개
  - 기술: 3개
  - 영업: 2개
  - 인사: 3개
  - 재무: 2개
  - 제품: 3개


In [ ]:
# [02-18] index_document 함수 정의
# [목적] 단일 문서를 임베딩 생성 후 OpenSearch에 저장하는 함수입니다
# [개념] 흐름: 제목+내용 합치기 → 임베딩 생성 → 원본+벡터 병합 → OpenSearch에 색인
def index_document(doc: dict, doc_id: str) -> None:
    """단일 문서 색인"""
    # 제목 + 내용을 합쳐서 임베딩 생성 (제목도 검색에 반영)
    text_for_embedding = f"{doc['title']}\n{doc['content']}"
    embedding = get_embedding(text_for_embedding)  # 텍스트 → 1536차원 벡터

    # 원본 문서에 임베딩 벡터와 생성일을 추가
    doc_with_embedding = {
        **doc,                         # 원본 필드 전부 복사 (title, content 등)
        "embedding": embedding,        # 임베딩 벡터 추가
        "created_at": "2024-01-01",    # 생성일 메타데이터
    }

    # OpenSearch에 색인 (저장)
    opensearch_client.index(
        index=INDEX_NAME,              # 저장할 인덱스
        id=doc_id,                     # 문서 고유 ID
        body=doc_with_embedding,       # 저장할 문서 본문
        refresh=True,                  # 즉시 검색 가능하도록 인덱스 갱신
    )

In [ ]:
# [02-19] 문서 색인 실행
# [목적] 전체 샘플 문서를 하나씩 OpenSearch에 색인(저장)합니다
# [주의] 문서 수만큼 OpenAI API가 호출됩니다 (비용 발생). 진행률이 표시됩니다
# 문서 색인 실행: 전체 문서를 순회하며 하나씩 임베딩 + 저장
print("문서 색인 중...\n")
for i, doc in enumerate(SAMPLE_DOCUMENTS):
    index_document(doc, doc_id=str(i + 1))  # 문서 ID: 1부터 순차 부여
    if (i + 1) % 10 == 0 or i == len(SAMPLE_DOCUMENTS) - 1:  # 10건 단위 또는 마지막
        print(f"  진행: {i+1}/{len(SAMPLE_DOCUMENTS)}")

print(f"\n총 {len(SAMPLE_DOCUMENTS)}개 문서 색인 완료")

문서 색인 중...

  진행: 10/20
  진행: 20/20

총 20개 문서 색인 완료


In [ ]:
# [02-20] 색인 확인
# [목적] 인덱스에 저장된 문서 수가 샘플 데이터 수와 일치하는지 확인합니다
# 색인 확인: 인덱스에 저장된 전체 문서 수 조회
count = opensearch_client.count(index=INDEX_NAME)  # count API로 문서 수 반환
print(f"인덱스 '{INDEX_NAME}' 문서 수: {count['count']}")

인덱스 'student_00_company_docs' 문서 수: 20


## 2.7 벌크 인덱싱
대량 문서 처리 시 벌크 API 사용으로 성능 향상

In [ ]:
# [02-21] bulk_index_documents 함수 정의
# [목적] 여러 문서를 한 번에 색인하는 벌크(bulk) 인덱싱 함수입니다
# [개념] 벌크 API = 여러 문서를 하나의 HTTP 요청으로 처리. 대량 데이터에 효율적입니다
def bulk_index_documents(docs: list[dict], index_name: str) -> dict:
    """벌크 인덱싱"""
    # 모든 문서의 임베딩을 한 번에 생성 (배치 처리)
    texts = [f"{d['title']}\n{d['content']}" for d in docs]  # 제목+내용 합치기
    embeddings = get_embeddings_batch(texts)  # 한 번의 API 호출로 전부 임베딩

    # 벌크 요청 본문 구성: [액션, 문서, 액션, 문서, ...] 형태의 교대 리스트
    bulk_body = []
    for i, (doc, emb) in enumerate(zip(docs, embeddings)):
        # 액션 메타: 어떤 인덱스에 어떤 ID로 저장할지 지정
        bulk_body.append({"index": {"_index": index_name, "_id": str(i + 100)}})
        # 문서 본문: 원본 + 임베딩 + 메타데이터
        bulk_body.append({
            **doc,
            "embedding": emb,
            "created_at": "2024-01-01",
        })

    # 벌크 API 실행: 하나의 HTTP 요청으로 모든 문서를 색인
    response = opensearch_client.bulk(body=bulk_body, refresh=True)
    return response

In [ ]:
# [02-22] 벌크 인덱싱 동작 확인
# [목적] 벌크 인덱싱 함수가 정상적으로 정의되었는지 확인합니다 (실제 실행은 선택)
# 벌크 인덱싱은 대량 문서 처리 시 사용
# sample_data.py의 문서들은 이미 위에서 색인했으므로,
# 여기서는 벌크 인덱싱 함수 동작만 확인합니다.

# 새 문서 예시: 벌크 함수에 전달할 딕셔너리 리스트
test_docs = [
    {"title": "테스트 문서 1", "content": "벌크 인덱싱 테스트용 문서입니다.", "category": "테스트", "source": "test.pdf"},
    {"title": "테스트 문서 2", "content": "이 문서도 벌크로 색인됩니다.", "category": "테스트", "source": "test.pdf"},
]

# 별도 인덱스에 테스트 (실제 인덱스에 영향 없음)
print("벌크 인덱싱 함수 정의 완료 (필요시 사용)")

벌크 인덱싱 함수 정의 완료 (필요시 사용)


## 2.8 인덱스 조회 및 관리

In [ ]:
# [02-23] 전체 문서 조회
# [목적] 인덱스에 저장된 모든 문서의 ID, 제목, 카테고리를 조회합니다
# [개념] match_all = 조건 없이 모든 문서를 검색하는 쿼리입니다
# 전체 문서 조회: match_all 쿼리로 모든 문서 검색
response = opensearch_client.search(
    index=INDEX_NAME,
    body={"query": {"match_all": {}}, "size": 100}  # 최대 100건 조회
)

print(f"총 {response['hits']['total']['value']}개 문서:\n")
for hit in response['hits']['hits']:  # 검색 결과(hits) 순회
    src = hit['_source']  # 문서 원본 데이터
    print(f"  [{hit['_id']}] {src['title']} ({src['category']})")

총 20개 문서:

  [1] 사내 복지 제도 안내 (인사)
  [6] API 연동 가이드 (제품)
  [2] 신입사원 온보딩 프로그램 (인사)
  [9] 시스템 장애 대응 매뉴얼 (IT)
  [5] 클라우드 서비스 요금제 안내 (제품)
  [11] 고객사 관리 정책 (영업)
  [3] 휴가 및 근태 관리 규정 (인사)
  [10] 영업 프로세스 가이드 (영업)
  [8] 개발 환경 설정 가이드 (IT)
  [7] 정보보안 정책 (IT)
  [4] 2024년 신제품 출시 계획 (제품)
  [19] 머신러닝 모델 배포 가이드 (AI/ML)
  [15] 데이터베이스 설계 표준 (기술)
  [20] 자연어 처리 프로젝트 템플릿 (AI/ML)
  [17] 사내 교육 프로그램 안내 (교육)
  [13] 예산 수립 및 집행 가이드 (재무)
  [16] 코드 리뷰 가이드라인 (기술)
  [14] 마이크로서비스 아키텍처 가이드 (기술)
  [18] 리더십 개발 과정 (교육)
  [12] 경비 처리 규정 (재무)


In [ ]:
# [02-24] 특정 문서 조회
# [목적] 문서 ID로 특정 문서 한 건을 조회하여 저장 상태를 확인합니다
# [개념] get() = ID 기반 단건 조회. 임베딩이 1536차원으로 저장되었는지 확인합니다
# 특정 문서 조회: ID "1"인 문서를 직접 가져오기
doc = opensearch_client.get(index=INDEX_NAME, id="1")  # ID 기반 단건 조회
print(f"문서 제목: {doc['_source']['title']}")
print(f"카테고리: {doc['_source']['category']}")
print(f"임베딩 차원: {len(doc['_source']['embedding'])}")  # 1536이면 정상

문서 제목: 사내 복지 제도 안내
카테고리: 인사
임베딩 차원: 1536


## 2.9 정리
### 학습 내용

| 주제 | 핵심 내용 |
|------|----------|
| 인덱스 | 문서 집합, 매핑으로 필드 타입 정의 |
| 임베딩 | 텍스트를 벡터로 변환, 배치 처리로 효율화 |
| 청킹 | 긴 문서를 작은 조각으로 분할 |
| 인덱싱 | 문서 + 임베딩을 OpenSearch에 저장 |

### 다음 학습
- 키워드 검색 (BM25)
- Full Text Query vs Exact Query